**Setup**

Import the required libraries and datasets.

* Dataset: bitext/customer-support-llm-chatbot-training-dataset (Modified)
* Original: https://huggingface.co/datasets/bitext/customer-support-llm-chatbot-training-dataset

MODIFICATIONS (Educational):
- Added synthetic PII columns and injected PII into text
- Introduced data quality issues
- Purpose: Simulate realistic data preprocessing challenges

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Load datasets
offline_test = pd.read_csv('./data/offline_evaluation_test.csv')
ab_test = pd.read_csv('./data/ab_test_results.csv')

**Example**

You'll work with two datasets:

1. **Offline test set** - 500 conversations with human-labeled correct responses, tested with both Model A (GPT-3.5) and Model B (GPT-4)
2. **A/B test results** - Real customer interactions split between both models

Explore the offline test data:

In [2]:
print("=== Offline Test Dataset ===")
print(f"Total test conversations: {len(offline_test)}")
print(f"\nColumns: {list(offline_test.columns)}")
print(f"\nSample:")
print(offline_test[['conversation_id', 'ground_truth', 'model_a_correct', 'model_b_correct', 'model_a_hallucinated', 'model_b_hallucinated']].head(3))

=== Offline Test Dataset ===
Total test conversations: 500

Columns: ['conversation_id', 'ground_truth', 'model_a_correct', 'model_a_hallucinated', 'model_b_correct', 'model_b_hallucinated']

Sample:
  conversation_id  ground_truth  model_a_correct  model_b_correct  \
0     conv_009329          True            False             True   
1     conv_004160         False             True             True   
2     conv_018500          True            False            False   

   model_a_hallucinated  model_b_hallucinated  
0                 False                 False  
1                 False                 False  
2                  True                 False  


You're about to upgrade your AI chatbot from GPT-3.5 to GPT-4. You have 500 conversations where humans already labeled the correct responses. Both GPT-3.5 (Model A) and GPT-4 (Model B) attempted to answer these same questions. Now it's your job to figure out which one actually performs better in the offline evaluation.

1. Calculate the accuracy for Model A (GPT-3.5) - what % of responses were correct?
2. Calculate the accuracy for Model B (GPT-4)
3. Calculate the hallucination rate for both models (how often did they make stuff up?)

In [3]:
# Calculate accuracy for both models
model_a_accuracy = offline_test['model_a_correct'].mean()
model_b_accuracy = offline_test['model_b_correct'].mean()

# Calculate hallucination rates
model_a_hallucination = offline_test['model_a_hallucinated'].mean()
model_b_hallucination = offline_test['model_b_hallucinated'].mean()

# Display results
print("=== Offline Evaluation Results ===")
print(f"\nModel A (GPT-3.5):")
print(f"  Accuracy: {model_a_accuracy:.1%}")
print(f"  Hallucination rate: {model_a_hallucination:.1%}")

print(f"\nModel B (GPT-4):")
print(f"  Accuracy: {model_b_accuracy:.1%}")
print(f"  Hallucination rate: {model_b_hallucination:.1%}")

=== Offline Evaluation Results ===

Model A (GPT-3.5):
  Accuracy: 61.8%
  Hallucination rate: 8.0%

Model B (GPT-4):
  Accuracy: 63.6%
  Hallucination rate: 5.0%


Now that you've calculated the differences in accuracy and hallucination rate, it's time to see if those differences are statistically significant.

1. Run a statistical significance test using `stats.chi2_contingency()` to determine if the accuracy difference is real or just due to chance
   - The input to the Chi-squared test requires a contingency table, a 2×2 array with the counts of outcomes for each model (rows) and result (correct/incorrect columns)
2. Display the p-value and interpret the results

In [4]:
# Model specifications
# Create contingency table for chi-square test
# Rows: Model A, Model B | Columns: Correct, Incorrect
model_a_correct = offline_test['model_a_correct'].sum()
model_a_incorrect = offline_test['model_a_correct'].count() - model_a_correct
model_b_correct = offline_test['model_b_correct'].sum()
model_b_incorrect = offline_test['model_b_correct'].count() - model_b_correct

contingency_table = [
    [model_a_correct, model_a_incorrect],
    [model_b_correct, model_b_incorrect]
]

# Run chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)


print(f"\nStatistical Test:")
print(f"  Chi-square statistic: {chi2:.2f}")
print(f"  P-value: {p_value:.4f}")

if p_value < 0.05:
    print(f"\n The accuracy difference IS statistically significant (p < 0.05)")
    print(f"   Model B is likely genuinely better than Model A")
else:
    print(f"\nThe accuracy difference is NOT statistically significant (p >= 0.05)")
    print(f"   The difference might just be random chance")


Statistical Test:
  Chi-square statistic: 0.27
  P-value: 0.6009

The accuracy difference is NOT statistically significant (p >= 0.05)
   The difference might just be random chance


Offline evaluation suggested Model B is better in terms of accuracy, but the difference was _not_ statistically significant. So you ran an A/B test with real users. Now you need to analyze the results to make a final deployment decision.

1. Calculate the resolution rate for both variants (control = Model A, treatment = Model B)
2. Calculate the average costs for both variants
3. Run a Chi-squared test on the resolution rates to check significance
4. Print a comparison of the model metrics

In [5]:
## YOUR SOLUTION HERE ##

# Split data by variant
control = ab_test[ab_test['variant'] == 'control']
treatment = ab_test[ab_test['variant'] == 'treatment']

# Calculate resolution rates
control_resolution = control['resolved'].mean()
treatment_resolution = treatment['resolved'].mean()

# Calculate avg costs 
control_cost = control['cost'].mean()
treatment_cost = treatment['cost'].mean()

# Chi-square test for resolution rates
control_resolved = control['resolved'].sum()
control_unresolved = control['resolved'].count() - control_resolved
treatment_resolved = treatment['resolved'].sum()
treatment_unresolved = treatment['resolved'].count() - treatment_resolved

resolution_contingency = [
    [control_resolved, control_unresolved],
    [treatment_resolved, treatment_unresolved]
]

chi2_resolution, resolution_p_value, dof, expected = stats.chi2_contingency(resolution_contingency)

# Display results
print("=== A/B Test Results ===")
print(f"\nSample sizes:")
print(f"  Control (Model A): {len(control):,} conversations")
print(f"  Treatment (Model B): {len(treatment):,} conversations")

print(f"\nResolution Rates:")
print(f"  Control: {control_resolution:.1%}")
print(f"  Treatment: {treatment_resolution:.1%}")
print(f"  Difference: {(treatment_resolution - control_resolution):.1%}")
print(f"  P-value: {resolution_p_value:.4f}")

if resolution_p_value < 0.05:
    print(f"  Statistically significant difference")
else:
    print(f"   Not statistically significant")

print(f"\nCosts:")
print(f"  Control: ${control_cost:.4f}")
print(f"  Treatment: ${treatment_cost:.4f}")
print(f"  Difference %: {(treatment_cost - control_cost)/control_cost:.1%}")

=== A/B Test Results ===

Sample sizes:
  Control (Model A): 1,000 conversations
  Treatment (Model B): 1,000 conversations

Resolution Rates:
  Control: 75.8%
  Treatment: 78.8%
  Difference: 3.0%
  P-value: 0.1216
   Not statistically significant

Costs:
  Control: $0.0033
  Treatment: $0.0034
  Difference %: 2.1%
